# Course 15: Research Engineering Interviews
## Session 4: ML from scratch I

**Goal:** implement stable cross-entropy, logistic regression and k-means in NumPy, 20 minutes each.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### Why the naive softmax fails

In [ ]:
import numpy as np
z = np.array([1000.0, 1001.0, 999.0])
with np.errstate(over="ignore", invalid="ignore"):
    print("naive:", np.exp(z) / np.exp(z).sum())

### Tests
Write `log_softmax(z)`, `cross_entropy(logits, y)`, `fit_logreg(X, y, lam, lr, steps)` returning `(w, b)`, and `kmeans(X, k, steps, seed)` returning `(centers, labels)`.

In [ ]:
def make_blobs(n=100, seed=1):
    rng = np.random.default_rng(seed)
    X = np.concatenate([rng.normal(0, 0.5, (n, 2)), rng.normal(5, 0.5, (n, 2))])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    return X, y

def run_tests(log_softmax, cross_entropy, fit_logreg, kmeans):
    z = np.array([[1000.0, 1001.0, 999.0], [0.0, 0.0, 0.0]])
    assert np.allclose(np.exp(log_softmax(z)).sum(-1), 1.0)
    assert abs(cross_entropy(z, np.array([1, 2])) - 0.7531091) < 1e-5
    X, y = make_blobs()
    Xs = (X - X.mean(0)) / X.std(0)
    w, b = fit_logreg(Xs, y, lam=0.0, lr=0.5, steps=500)
    acc = (((Xs @ w + b) > 0) == y).mean()
    assert acc > 0.99, acc
    centers, labels = kmeans(X, 2, steps=20, seed=0)
    c = centers[np.argsort(centers[:, 0])]
    assert np.allclose(c, [[0, 0], [5, 5]], atol=0.3) and sorted(np.bincount(labels)) == [100, 100]
    print("all tests pass | logistic accuracy", acc, "| centres", np.round(c, 2).tolist())

In [ ]:
def log_softmax(z): raise NotImplementedError                       # TODO (15 min)
def cross_entropy(logits, y): raise NotImplementedError             # TODO
def fit_logreg(X, y, lam=0.0, lr=0.1, steps=1000): raise NotImplementedError   # TODO (20 min)
def kmeans(X, k, steps=20, seed=0): raise NotImplementedError       # TODO (20 min)

# run_tests(log_softmax, cross_entropy, fit_logreg, kmeans)

### Follow-up questions to answer aloud
- Why subtract the maximum logit?
- What happens to logistic regression's weights on perfectly separable data without regularization?
- What if a k-means cluster becomes empty? Does k-means always converge? How do you choose k?

---
## Reference solutions
Try every problem under the timer first. Then run this section: it defines working versions and runs the same tests.

In [ ]:
def log_softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))

def cross_entropy(logits, y):
    return -log_softmax(logits)[np.arange(len(y)), y].mean()

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fit_logreg(X, y, lam=0.0, lr=0.1, steps=1000):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)                       # (n,)
        w -= lr * (X.T @ (p - y) / len(y) + lam * w)   # (d,)
        b -= lr * np.mean(p - y)
    return w, b

def kmeans(X, k, steps=20, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)]
    for _ in range(steps):
        d = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(-1)      # (n, k)
        labels = d.argmin(1)
        centers = np.stack([X[labels == j].mean(0) if (labels == j).any() else centers[j] for j in range(k)])
    return centers, labels

run_tests(log_softmax, cross_entropy, fit_logreg, kmeans)